# Image Classification Pattern

Follow an image batch from dataset contract through a small CNN, evaluation, and filename-keyed inference.

- **Study time:** 50-70 minutes
- **Prerequisites:** PyTorch DataLoader, convolution shapes, and the canonical training loop
- **Mode:** `optional`
- **Data policy:** no downloads or image files; deterministic circle/square tensors are generated in memory
- **Provenance:** consolidated from the legacy CIFAR case study and portal dataset-pattern notebooks

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the DataCoding project")


PROJECT_ROOT = find_project_root()
source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


def seed_all(seed=61):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


seed_all()
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
show("Environment | selected device", device)

## 1. Dataset returns `(C, H, W)`, integer label, and stable identifier


In [ ]:
class ShapeDataset(Dataset):
    classes = ("circle", "square")

    def __init__(self, n_samples=600, image_size=20, seed=61):
        generator = torch.Generator().manual_seed(seed)
        self.labels = torch.arange(n_samples) % 2
        self.filenames = [f"shape_{index:04d}.png" for index in range(n_samples)]
        coordinates = torch.linspace(-1, 1, image_size)
        yy, xx = torch.meshgrid(coordinates, coordinates, indexing="ij")
        circle = ((xx**2 + yy**2) <= 0.48**2).float()
        square = ((xx.abs() <= 0.48) & (yy.abs() <= 0.48)).float()
        templates = torch.stack([circle, square])[:, None, :, :]
        noise = 0.12 * torch.randn(n_samples, 1, image_size, image_size, generator=generator)
        self.images = (templates[self.labels] + noise).clamp(0, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.images[index], self.labels[index], self.filenames[index]


dataset = ShapeDataset()
train_dataset, validation_dataset, test_dataset = random_split(
    dataset,
    [420, 90, 90],
    generator=torch.Generator().manual_seed(61),
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
sample_X, sample_y, sample_names = next(iter(train_loader))

show("Dataset | batch NCHW shape", tuple(sample_X.shape))
show(
    "Dataset | batch dtype and min/max",
    (sample_X.dtype, sample_X.min().item(), sample_X.max().item()),
)
show("Dataset | first labels and identifiers", (sample_y[:5].tolist(), list(sample_names[:5])))

## 2. Small CNN produces one logit per class


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(16, n_classes)

    def forward(self, X):
        features = self.features(X).flatten(1)
        return self.classifier(features)


model = SmallCNN().to(device)
with torch.no_grad():
    sample_logits = model(sample_X.to(device))
show("Model | input and logits shapes", (tuple(sample_X.shape), tuple(sample_logits.shape)))

## 3. Train/evaluate without augmenting validation


In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for X_batch, y_batch, _ in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(X_batch)
            total_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total_examples += len(X_batch)
    return total_loss / total_examples, total_correct / total_examples


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1, 7):
    train_loss, train_accuracy = run_epoch(model, train_loader, loss_fn, optimizer)
    validation_loss, validation_accuracy = run_epoch(model, validation_loader, loss_fn)
    message = (
        f"epoch={epoch:02d} train_loss={train_loss:.4f} "
        f"train_acc={train_accuracy:.3f} validation_loss={validation_loss:.4f} "
        f"validation_acc={validation_accuracy:.3f}"
    )
    show("Training | epoch metrics", message)

## 4. Inference remains keyed by filename


In [ ]:
model.eval()
rows = []
with torch.no_grad():
    for X_batch, y_batch, names in test_loader:
        probabilities = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        predictions = probabilities.argmax(dim=1)
        for name, target, prediction, confidence in zip(
            names,
            y_batch,
            predictions,
            probabilities.max(dim=1).values,
            strict=True,
        ):
            rows.append(
                {
                    "filename": name,
                    "target": dataset.classes[target.item()],
                    "prediction": dataset.classes[prediction.item()],
                    "confidence": confidence.item(),
                }
            )

inference = pd.DataFrame(rows)
test_accuracy = (inference["target"] == inference["prediction"]).mean()
show("Inference | first five filename-keyed predictions", inference.head().to_string(index=False))
show("Inference | test accuracy", test_accuracy)
show("Image checks | status", "NCHW, logits, model modes, and identifier mapping verified")